# Module 1.3: Prompt Optimization for Agentic AI

<span class="badge blue">Agentic AI</span> <span class="badge amber">~15 min</span> <span class="badge mint">Hands-on</span>

## Learning Objectives
By completing this notebook, you will:
1. Implement few-shot learning to teach agents through examples
2. Design reusable prompt templates with variable substitution
3. Build dynamic prompting systems that adapt based on context
4. Measure and optimize prompt effectiveness systematically
5. Apply prompt compression techniques for cost reduction

## Prerequisites
- Module 1.1: System Prompt Design
- Module 1.2: Reasoning Patterns
- Understanding of string formatting and templates

## Estimated Time: 50 minutes

---

## Why Prompt Optimization Matters

Prompt engineering is the primary interface for controlling agent behavior. Well-optimized prompts:

- **Reduce error rates** by providing clear examples and constraints
- **Lower costs** by minimizing token usage while maintaining quality
- **Enable consistency** across multiple agent invocations
- **Facilitate maintenance** through reusable, modular designs

**Real-world application:** Production systems use optimized prompt templates to ensure consistent behavior across thousands of agent calls while keeping costs predictable.

**Key insight:** A 10% improvement in prompt design can yield 50%+ improvement in task accuracy and 30% reduction in costs.

## Setup & Imports

In [ ]:
# Setup: Course styling and plot theme
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname("__file__"), "../../../../.."))

from resources.notebook_style import apply_course_theme
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()
print("Course theme applied.")

In [ ]:
# Purpose: Import libraries for prompt optimization and templating
# Key Concept: Systematic prompt engineering requires measurement and iteration

import os
import json
import re
from typing import Dict, List, Any, Optional, Callable
from dataclasses import dataclass, field
from string import Template
from openai import OpenAI
from pydantic import BaseModel, Field
import tiktoken

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

MODEL = "gpt-4o-mini"
TEMPERATURE = 0.0

# Token counting
encoding = tiktoken.encoding_for_model(MODEL)

def count_tokens(text: str) -> int:
    """Count tokens in text."""
    return len(encoding.encode(text))

print("✓ Environment configured for prompt optimization")

## Part 1: Few-Shot Learning - Teaching Through Examples

Few-shot learning provides the agent with examples of desired input-output behavior. This is more reliable than instructions alone for:

- **Complex formatting** requirements
- **Nuanced judgments** that are hard to specify
- **Domain-specific** tasks with specialized patterns

In [ ]:
# Purpose: Implement few-shot learning with structured examples
# Key Concept: Examples teach patterns that are difficult to specify explicitly

@dataclass
class Example:
    """A single few-shot example."""
    input: str
    output: str
    explanation: Optional[str] = None


def create_few_shot_prompt(
    task_description: str,
    examples: List[Example],
    user_input: str,
    include_explanations: bool = False
) -> str:
    """
    Build a few-shot prompt with examples.
    
    Parameters
    ----------
    task_description : str
        High-level description of the task
    examples : List[Example]
        Training examples
    user_input : str
        Current input to process
    include_explanations : bool
        Whether to include reasoning in examples
    
    Returns
    -------
    str
        Complete few-shot prompt
    """
    parts = [task_description, ""]
    
    # Add examples
    for i, ex in enumerate(examples, 1):
        parts.append(f"Example {i}:")
        parts.append(f"Input: {ex.input}")
        
        if include_explanations and ex.explanation:
            parts.append(f"Reasoning: {ex.explanation}")
        
        parts.append(f"Output: {ex.output}")
        parts.append("")
    
    # Add user input
    parts.append("Now complete this:")
    parts.append(f"Input: {user_input}")
    parts.append("Output:")
    
    return "\n".join(parts)


# Example: Intent classification for customer support
intent_examples = [
    Example(
        input="I can't log into my account",
        output='{"intent": "authentication_issue", "urgency": "medium", "department": "technical_support"}',
        explanation="User experiencing login problems - technical issue requiring support"
    ),
    Example(
        input="How much does the premium plan cost?",
        output='{"intent": "pricing_inquiry", "urgency": "low", "department": "sales"}',
        explanation="Question about pricing - sales inquiry with no urgency"
    ),
    Example(
        input="URGENT: Unauthorized charge on my card!",
        output='{"intent": "billing_dispute", "urgency": "high", "department": "billing"}',
        explanation="Billing issue marked urgent - immediate attention required"
    )
]

test_input = "The API is returning 500 errors for all my requests"

few_shot_prompt = create_few_shot_prompt(
    task_description="Classify customer support messages into structured JSON with intent, urgency, and department.",
    examples=intent_examples,
    user_input=test_input,
    include_explanations=False
)

print("Few-Shot Prompt:")
print("=" * 60)
print(few_shot_prompt)
print("\n" + "=" * 60)
print(f"Token count: {count_tokens(few_shot_prompt)}")

In [ ]:
# Purpose: Test few-shot effectiveness
# Key Concept: Examples enable consistent structured outputs

def call_with_prompt(prompt: str) -> str:
    """Call LLM with prompt."""
    response = client.chat.completions.create(
        model=MODEL,
        temperature=TEMPERATURE,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


# Compare zero-shot vs few-shot
zero_shot_prompt = f"""Classify this customer message into JSON with intent, urgency, and department:
Message: {test_input}
Output:"""

print("\nZero-Shot Result:")
print("-" * 60)
zero_shot_result = call_with_prompt(zero_shot_prompt)
print(zero_shot_result)

print("\n\nFew-Shot Result:")
print("-" * 60)
few_shot_result = call_with_prompt(few_shot_prompt)
print(few_shot_result)

# Validate JSON
try:
    json.loads(few_shot_result)
    print("\n✓ Few-shot output is valid JSON")
except:
    print("\n✗ Few-shot output is not valid JSON")

## Part 2: Prompt Templates - Reusable Patterns

Templates separate prompt structure from variable content, enabling:

- **Reusability** across different inputs
- **Maintainability** by centralizing prompt logic
- **Version control** of prompt evolution
- **A/B testing** of prompt variants

In [ ]:
# Purpose: Build a prompt template system with validation
# Key Concept: Templates enable systematic prompt management

class PromptTemplate:
    """
    A reusable prompt template with variable substitution.
    """
    
    def __init__(
        self,
        template: str,
        required_vars: List[str],
        optional_vars: Optional[Dict[str, str]] = None,
        validators: Optional[Dict[str, Callable]] = None
    ):
        """
        Initialize prompt template.
        
        Parameters
        ----------
        template : str
            Template string with ${variable} placeholders
        required_vars : List[str]
            Variables that must be provided
        optional_vars : Optional[Dict[str, str]]
            Default values for optional variables
        validators : Optional[Dict[str, Callable]]
            Validation functions for variables
        """
        self.template = Template(template)
        self.required_vars = set(required_vars)
        self.optional_vars = optional_vars or {}
        self.validators = validators or {}
    
    def render(self, **kwargs) -> str:
        """
        Render template with provided variables.
        
        Parameters
        ----------
        **kwargs : Any
            Variable values
        
        Returns
        -------
        str
            Rendered prompt
        
        Raises
        ------
        ValueError
            If required variables missing or validation fails
        """
        # Check required variables
        provided = set(kwargs.keys())
        missing = self.required_vars - provided
        if missing:
            raise ValueError(f"Missing required variables: {missing}")
        
        # Add optional defaults
        values = {**self.optional_vars, **kwargs}
        
        # Validate
        for var, validator in self.validators.items():
            if var in values:
                if not validator(values[var]):
                    raise ValueError(f"Validation failed for variable: {var}")
        
        # Render
        return self.template.safe_substitute(values)
    
    def get_token_count(self, **kwargs) -> int:
        """Get token count for rendered template."""
        rendered = self.render(**kwargs)
        return count_tokens(rendered)


# Example: Email response template
email_response_template = PromptTemplate(
    template="""You are a ${role} for ${company}.

Customer email:
${customer_email}

Context: ${context}

Write a ${tone} response that:
- Addresses the customer's concerns
- ${additional_instructions}
- Maintains a ${tone} tone
- Keeps response under ${max_length} words

Response:""",
    required_vars=["customer_email"],
    optional_vars={
        "role": "Customer Support Representative",
        "company": "our company",
        "context": "No additional context provided",
        "tone": "professional and empathetic",
        "additional_instructions": "Provide clear next steps",
        "max_length": "150"
    },
    validators={
        "customer_email": lambda x: len(x) > 10,
        "max_length": lambda x: x.isdigit() and int(x) > 0
    }
)

# Test template
test_email = "I've been waiting 3 days for a response to my refund request. This is unacceptable!"

prompt = email_response_template.render(
    customer_email=test_email,
    company="TechCorp",
    context="Customer has valid refund request from 3 days ago, ticket #12345",
    tone="apologetic and action-oriented",
    max_length="100"
)

print("Rendered Template:")
print("=" * 60)
print(prompt)
print(f"\nToken count: {email_response_template.get_token_count(customer_email=test_email)}")

## Part 3: Dynamic Prompting - Context-Aware Adaptation

Dynamic prompts adapt based on:
- **Input characteristics** (length, complexity, language)
- **User context** (expertise level, history)
- **System state** (available tools, resource constraints)

In [ ]:
# Purpose: Implement dynamic prompt generation based on context
# Key Concept: Adaptive prompts optimize for different scenarios

@dataclass
class UserContext:
    """Context about the user for prompt adaptation."""
    expertise_level: str  # "beginner", "intermediate", "expert"
    preferred_language: str
    interaction_history: List[str] = field(default_factory=list)
    constraints: Dict[str, Any] = field(default_factory=dict)


def create_dynamic_prompt(
    base_task: str,
    user_input: str,
    context: UserContext,
    available_examples: List[Example]
) -> str:
    """
    Generate a prompt dynamically adapted to user context.
    
    Parameters
    ----------
    base_task : str
        Core task description
    user_input : str
        User's query or input
    context : UserContext
        User context for adaptation
    available_examples : List[Example]
        Pool of examples to choose from
    
    Returns
    -------
    str
        Dynamically generated prompt
    """
    parts = []
    
    # Adapt task description based on expertise
    if context.expertise_level == "beginner":
        parts.append(f"{base_task}\n\nProvide detailed explanations suitable for beginners.")
    elif context.expertise_level == "expert":
        parts.append(f"{base_task}\n\nProvide concise, technical responses.")
    else:
        parts.append(base_task)
    
    parts.append("")
    
    # Add examples if user is beginner or input is complex
    input_complexity = len(user_input.split())
    if context.expertise_level == "beginner" or input_complexity > 50:
        # Select most relevant examples
        num_examples = 2 if context.expertise_level == "beginner" else 1
        selected_examples = available_examples[:num_examples]
        
        if selected_examples:
            parts.append("Examples:")
            for i, ex in enumerate(selected_examples, 1):
                parts.append(f"\n{i}. Input: {ex.input}")
                parts.append(f"   Output: {ex.output}")
            parts.append("")
    
    # Add conversation history for context
    if context.interaction_history:
        recent = context.interaction_history[-2:]  # Last 2 interactions
        parts.append("Previous context:")
        parts.extend(recent)
        parts.append("")
    
    # Add constraints
    if context.constraints:
        constraint_text = ", ".join(f"{k}: {v}" for k, v in context.constraints.items())
        parts.append(f"Constraints: {constraint_text}")
        parts.append("")
    
    # Add user input
    parts.append(f"User query: {user_input}")
    parts.append("\nResponse:")
    
    return "\n".join(parts)


# Test dynamic prompting with different contexts
base_task = "Explain how to optimize database queries."
user_query = "My SELECT query is slow"

# Beginner context
beginner_context = UserContext(
    expertise_level="beginner",
    preferred_language="en",
    constraints={"response_length": "detailed"}
)

# Expert context
expert_context = UserContext(
    expertise_level="expert",
    preferred_language="en",
    interaction_history=["Previous query was about indexing strategies"],
    constraints={"response_length": "concise"}
)

examples = [
    Example(
        input="How do I make my query faster?",
        output="Add indexes on frequently queried columns, use EXPLAIN to analyze query plan, avoid SELECT *"
    )
]

print("Dynamic Prompt (Beginner):")
print("=" * 60)
beginner_prompt = create_dynamic_prompt(base_task, user_query, beginner_context, examples)
print(beginner_prompt)
print(f"\nTokens: {count_tokens(beginner_prompt)}")

print("\n\n" + "=" * 60)
print("Dynamic Prompt (Expert):")
print("=" * 60)
expert_prompt = create_dynamic_prompt(base_task, user_query, expert_context, examples)
print(expert_prompt)
print(f"\nTokens: {count_tokens(expert_prompt)}")

## Part 4: Prompt Optimization - Measurement and Improvement

Systematic optimization requires:
1. **Metrics** to evaluate prompt quality
2. **Test cases** representing real scenarios
3. **Iterative refinement** based on failures
4. **Version tracking** for reproducibility

In [ ]:
# Purpose: Build a prompt optimization framework
# Key Concept: Systematic testing reveals prompt weaknesses

@dataclass
class TestCase:
    """A test case for prompt evaluation."""
    input: str
    expected_output: Optional[str] = None
    evaluation_criteria: Optional[Callable[[str], Dict[str, float]]] = None


def evaluate_prompt(
    prompt_fn: Callable[[str], str],
    test_cases: List[TestCase],
    verbose: bool = False
) -> Dict[str, Any]:
    """
    Evaluate a prompt function on test cases.
    
    Parameters
    ----------
    prompt_fn : Callable[[str], str]
        Function that takes input and returns generated prompt
    test_cases : List[TestCase]
        Test cases to evaluate
    verbose : bool
        Print detailed results
    
    Returns
    -------
    Dict[str, Any]
        Evaluation metrics
    """
    results = []
    total_tokens = 0
    total_score = 0.0
    
    for i, test_case in enumerate(test_cases, 1):
        # Generate prompt
        prompt = prompt_fn(test_case.input)
        tokens = count_tokens(prompt)
        total_tokens += tokens
        
        # Call LLM
        output = call_with_prompt(prompt)
        
        # Evaluate
        if test_case.evaluation_criteria:
            scores = test_case.evaluation_criteria(output)
            avg_score = sum(scores.values()) / len(scores)
        else:
            # Simple exact match
            avg_score = 1.0 if output.strip() == test_case.expected_output else 0.0
            scores = {"exact_match": avg_score}
        
        total_score += avg_score
        
        result = {
            'test_case': i,
            'input': test_case.input,
            'output': output,
            'scores': scores,
            'avg_score': avg_score,
            'tokens': tokens
        }
        results.append(result)
        
        if verbose:
            print(f"\nTest {i}:")
            print(f"Input: {test_case.input[:50]}...")
            print(f"Score: {avg_score:.2f}")
            print(f"Tokens: {tokens}")
    
    return {
        'results': results,
        'avg_score': total_score / len(test_cases),
        'avg_tokens': total_tokens / len(test_cases),
        'total_tokens': total_tokens
    }


# Example evaluation criteria
def json_format_check(output: str) -> Dict[str, float]:
    """Check if output is valid JSON with required fields."""
    scores = {}
    
    # Valid JSON?
    try:
        data = json.loads(output)
        scores['valid_json'] = 1.0
        
        # Has required fields?
        required = ['intent', 'urgency', 'department']
        has_fields = all(field in data for field in required)
        scores['has_required_fields'] = 1.0 if has_fields else 0.0
        
        # Valid values?
        valid_urgency = data.get('urgency') in ['low', 'medium', 'high']
        scores['valid_values'] = 1.0 if valid_urgency else 0.5
        
    except json.JSONDecodeError:
        scores['valid_json'] = 0.0
        scores['has_required_fields'] = 0.0
        scores['valid_values'] = 0.0
    
    return scores


# Create test cases
test_cases = [
    TestCase(
        input="I forgot my password",
        evaluation_criteria=json_format_check
    ),
    TestCase(
        input="Do you offer student discounts?",
        evaluation_criteria=json_format_check
    ),
    TestCase(
        input="URGENT: System is down!",
        evaluation_criteria=json_format_check
    )
]

# Define prompt function to test
def intent_classifier_v1(user_input: str) -> str:
    """Version 1: Simple instruction."""
    return f"""Classify this message into JSON with intent, urgency (low/medium/high), and department:
Message: {user_input}
JSON:"""

def intent_classifier_v2(user_input: str) -> str:
    """Version 2: With few-shot examples."""
    return create_few_shot_prompt(
        "Classify customer support messages into JSON.",
        intent_examples[:2],  # Use 2 examples
        user_input
    )

# Evaluate both versions
print("Evaluating Prompt V1 (Simple):")
print("=" * 60)
eval_v1 = evaluate_prompt(intent_classifier_v1, test_cases, verbose=False)
print(f"Average Score: {eval_v1['avg_score']:.2f}")
print(f"Average Tokens: {eval_v1['avg_tokens']:.0f}")

print("\n\nEvaluating Prompt V2 (Few-shot):")
print("=" * 60)
eval_v2 = evaluate_prompt(intent_classifier_v2, test_cases, verbose=False)
print(f"Average Score: {eval_v2['avg_score']:.2f}")
print(f"Average Tokens: {eval_v2['avg_tokens']:.0f}")

print("\n\nComparison:")
print("=" * 60)
score_improvement = (eval_v2['avg_score'] - eval_v1['avg_score']) / eval_v1['avg_score'] * 100
token_increase = (eval_v2['avg_tokens'] - eval_v1['avg_tokens']) / eval_v1['avg_tokens'] * 100
print(f"Score improvement: {score_improvement:+.1f}%")
print(f"Token increase: {token_increase:+.1f}%")

## Exercises

### Exercise 3.1: Build an Example Selector

**Task:** Create a function that selects the most relevant few-shot examples for a given input.

**Requirements:**
- Select examples based on similarity to input (simple keyword matching is fine)
- Limit total token budget for examples
- Return top N most relevant examples

**Expected Output:** List of selected examples within token budget

In [ ]:
# YOUR CODE HERE
# ---------------

def select_relevant_examples(
    user_input: str,
    available_examples: List[Example],
    max_examples: int = 3,
    token_budget: int = 500
) -> List[Example]:
    """
    Select most relevant examples for user input.
    
    Parameters
    ----------
    user_input : str
        User's query
    available_examples : List[Example]
        Pool of examples
    max_examples : int
        Maximum number of examples
    token_budget : int
        Maximum tokens for all examples
    
    Returns
    -------
    List[Example]
        Selected examples
    """
    # Replace with your implementation
    return []


# Test
# test_input = "How do I reset my password?"
# selected = select_relevant_examples(test_input, intent_examples)
# print(f"Selected {len(selected)} examples")
# for ex in selected:
#     print(f"  - {ex.input}")

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_3_1():
    examples = [
        Example("password reset", "out1"),
        Example("billing question", "out2"),
        Example("login issue", "out3")
    ]
    
    result = select_relevant_examples("can't log in", examples, max_examples=2)
    assert isinstance(result, list), "Should return a list"
    assert len(result) <= 2, "Should respect max_examples"
    assert all(isinstance(ex, Example) for ex in result), "Should return Example objects"
    
    # Check token budget
    total_tokens = sum(count_tokens(f"{ex.input}{ex.output}") for ex in result)
    assert total_tokens <= 500, "Should respect token budget"
    
    print("✅ Exercise 3.1 passed!")

test_exercise_3_1()

### Exercise 3.2: Implement Prompt Compression

**Task:** Create a function that compresses a prompt while preserving meaning.

**Requirements:**
- Remove redundant phrases
- Shorten verbose explanations
- Keep critical information intact
- Report compression ratio

**Expected Output:** Compressed prompt and token savings

In [ ]:
# YOUR CODE HERE
# ---------------

def compress_prompt(prompt: str) -> Dict[str, Any]:
    """
    Compress a prompt while preserving meaning.
    
    Parameters
    ----------
    prompt : str
        Original prompt
    
    Returns
    -------
    Dict[str, Any]
        Keys: 'compressed', 'original_tokens', 'compressed_tokens', 'savings_pct'
    """
    # Replace with your implementation
    return {
        'compressed': prompt,
        'original_tokens': 0,
        'compressed_tokens': 0,
        'savings_pct': 0.0
    }


# Test
# verbose_prompt = '''You are a helpful assistant. Please help the user.
# The user is asking about password reset. You should provide clear instructions.
# Make sure to be polite and professional in your response.'''
# result = compress_prompt(verbose_prompt)
# print(f"Original: {result['original_tokens']} tokens")
# print(f"Compressed: {result['compressed_tokens']} tokens")
# print(f"Savings: {result['savings_pct']:.1f}%")
# print(f"\nCompressed prompt:\n{result['compressed']}")

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_3_2():
    test_prompt = "Please help me. I need assistance. Can you assist me please? Thank you."
    result = compress_prompt(test_prompt)
    
    assert isinstance(result, dict), "Should return a dictionary"
    assert 'compressed' in result, "Should have 'compressed' key"
    assert 'original_tokens' in result, "Should have 'original_tokens' key"
    assert 'compressed_tokens' in result, "Should have 'compressed_tokens' key"
    assert 'savings_pct' in result, "Should have 'savings_pct' key"
    assert result['compressed_tokens'] <= result['original_tokens'], "Compressed should have fewer tokens"
    assert len(result['compressed']) > 0, "Compressed prompt should not be empty"
    
    print("✅ Exercise 3.2 passed!")

test_exercise_3_2()

### Exercise 3.3: Create an A/B Testing Framework

**Task:** Build a system to compare multiple prompt variants.

**Requirements:**
- Accept list of prompt functions
- Run each on same test cases
- Calculate win rate for each variant
- Identify statistically significant differences

**Expected Output:** Comparison report with winner recommendation

In [ ]:
# YOUR CODE HERE
# ---------------

def ab_test_prompts(
    variants: Dict[str, Callable[[str], str]],
    test_cases: List[TestCase]
) -> Dict[str, Any]:
    """
    A/B test multiple prompt variants.
    
    Parameters
    ----------
    variants : Dict[str, Callable[[str], str]]
        Named prompt functions to compare
    test_cases : List[TestCase]
        Test cases to evaluate
    
    Returns
    -------
    Dict[str, Any]
        Keys: 'variant_results', 'winner', 'recommendation'
    """
    # Replace with your implementation
    return {
        'variant_results': {},
        'winner': '',
        'recommendation': ''
    }


# Test
# variants = {
#     'v1_simple': intent_classifier_v1,
#     'v2_fewshot': intent_classifier_v2
# }
# results = ab_test_prompts(variants, test_cases[:2])
# print(f"Winner: {results['winner']}")
# print(f"Recommendation: {results['recommendation']}")

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_3_3():
    def variant_a(x): return f"Classify: {x}"
    def variant_b(x): return f"Classify this message: {x}"
    
    variants = {'A': variant_a, 'B': variant_b}
    test_cases_simple = [
        TestCase("test input", evaluation_criteria=lambda x: {'score': 0.8})
    ]
    
    result = ab_test_prompts(variants, test_cases_simple)
    assert isinstance(result, dict), "Should return a dictionary"
    assert 'variant_results' in result, "Should have 'variant_results' key"
    assert 'winner' in result, "Should have 'winner' key"
    assert result['winner'] in ['A', 'B'], "Winner should be one of the variants"
    
    print("✅ Exercise 3.3 passed!")

test_exercise_3_3()

### Exercise 3.4: Build a Prompt Library

**Task:** Create a system for storing and retrieving reusable prompt templates.

**Requirements:**
- Store templates with metadata (name, description, tags, version)
- Search templates by tags or description
- Track usage statistics
- Support versioning

**Expected Output:** Prompt library class with CRUD operations

In [ ]:
# YOUR CODE HERE
# ---------------

class PromptLibrary:
    """
    Library for managing reusable prompt templates.
    """
    
    def __init__(self):
        self.templates = {}
    
    def add_template(
        self,
        name: str,
        template: PromptTemplate,
        description: str,
        tags: List[str],
        version: str = "1.0"
    ) -> None:
        """Add a template to the library."""
        pass  # Replace with your implementation
    
    def get_template(self, name: str, version: Optional[str] = None) -> Optional[PromptTemplate]:
        """Retrieve a template by name."""
        pass  # Replace with your implementation
    
    def search_templates(self, query: str) -> List[str]:
        """Search templates by tags or description."""
        pass  # Replace with your implementation
    
    def track_usage(self, name: str) -> None:
        """Record template usage."""
        pass  # Replace with your implementation


# Test
# library = PromptLibrary()
# library.add_template(
#     "email_response",
#     email_response_template,
#     "Generate customer email responses",
#     ["email", "customer_support", "response"]
# )
# results = library.search_templates("email")
# print(f"Found {len(results)} templates")

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_3_4():
    library = PromptLibrary()
    
    # Test adding template
    test_template = PromptTemplate("Test ${var}", ["var"])
    library.add_template("test", test_template, "A test", ["testing"])
    
    # Test retrieval
    retrieved = library.get_template("test")
    assert retrieved is not None, "Should retrieve added template"
    
    # Test search
    results = library.search_templates("test")
    assert isinstance(results, list), "Search should return a list"
    assert len(results) > 0, "Should find matching templates"
    
    print("✅ Exercise 3.4 passed!")

test_exercise_3_4()

## Summary

**Key Takeaways:**

1. **Few-shot learning** teaches complex patterns more reliably than instructions alone
2. **Prompt templates** enable reusability, maintainability, and version control
3. **Dynamic prompting** adapts to user context, improving relevance and efficiency
4. **Systematic optimization** requires metrics, test cases, and iterative refinement
5. **Token efficiency** matters for cost at scale - compress without losing quality

**Best Practices:**
- Start with simple prompts, add complexity only when needed
- Maintain a library of tested templates for common tasks
- Always measure prompt performance on representative test cases
- Version control prompts like code
- Balance quality vs. cost based on use case criticality

**What's Next:**
- Module 2: Tool Use & Function Calling - Extending agents with external capabilities
- Module 3: Memory & Context - Maintaining state across interactions

**Additional Resources:**
- [Prompt Engineering Guide](https://www.promptingguide.ai/)
- [OpenAI Prompt Engineering Best Practices](https://platform.openai.com/docs/guides/prompt-engineering)
- [LangChain Prompt Templates](https://python.langchain.com/docs/modules/model_io/prompts/)